# Week05 Part 1 완성본 — Teachable Machine 결과 모델 활용

## 실행 순서
Notion의 **Part 1 실습 1 → 2 → 3 → 4** 순서와 이 노트북의 코드 셀이 1:1로 대응합니다.

### 준비 파일
- Teachable Machine에서 Export한 `keras_model.h5` 또는 `.keras`
- `labels.txt`
- 테스트 이미지 3장 이상

> 모델 파일이 없거나 로드가 되지 않으면 Part 2 노트북으로 이동하세요.


## 실습 1. Teachable Machine 모델 파일과 labels.txt 업로드


In [ ]:
from google.colab import files

uploaded_model_files = files.upload()
print("업로드된 파일:", list(uploaded_model_files.keys()))


## 실습 2. 모델과 라벨 불러오기

아래 코드는 업로드된 파일 중 `.h5` 또는 `.keras` 파일을 자동으로 찾고,
`labels.txt`를 읽습니다.


In [ ]:
import tensorflow as tf
from pathlib import Path

uploaded_names = list(uploaded_model_files.keys())

model_candidates = [n for n in uploaded_names if n.endswith((".h5", ".keras"))]
if not model_candidates:
    raise FileNotFoundError("Teachable Machine 모델(.h5 또는 .keras)을 업로드하세요.")

MODEL_FILE = model_candidates[0]

label_candidates = [n for n in uploaded_names if n.lower().endswith(".txt")]
if not label_candidates:
    raise FileNotFoundError("labels.txt를 업로드하세요.")

LABEL_FILE = label_candidates[0]

print("MODEL_FILE =", MODEL_FILE)
print("LABEL_FILE =", LABEL_FILE)

model = tf.keras.models.load_model(MODEL_FILE, compile=False)

with open(LABEL_FILE, "r", encoding="utf-8") as f:
    raw_labels = [line.strip() for line in f if line.strip()]

# Teachable Machine labels.txt가 "0 Class A" 형식인 경우 번호 제거
class_names = []
for label in raw_labels:
    parts = label.split(" ", 1)
    class_names.append(parts[1] if len(parts) == 2 and parts[0].isdigit() else label)

print("클래스:", class_names)
print("모델 입력 크기:", model.input_shape)


## 실습 3. 테스트 이미지 업로드


In [ ]:
test_uploaded = files.upload()
test_image_files = list(test_uploaded.keys())
print("테스트 이미지:", test_image_files)


## 실습 4. 이미지 전처리 함수 작성

Teachable Machine TensorFlow/Keras Export의 대표적인 정규화 방식인
`(pixel / 127.5) - 1`을 사용합니다.


In [ ]:
import numpy as np
from PIL import Image

height = int(model.input_shape[1])
width = int(model.input_shape[2])

def prepare_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = image.resize((width, height))
    array = np.asarray(image).astype(np.float32)
    array = (array / 127.5) - 1.0
    return np.expand_dims(array, axis=0)


## 실습 5. 테스트 이미지 1장 예측


In [ ]:
IMAGE_FILE = test_image_files[0]

x = prepare_image(IMAGE_FILE)
pred = model.predict(x, verbose=0)[0]
best_idx = int(np.argmax(pred))

print("이미지:", IMAGE_FILE)
print("예측 클래스:", class_names[best_idx])
print("Confidence:", float(pred[best_idx]))

for name, score in zip(class_names, pred):
    print(f"{name}: {score:.4f}")


## 실습 6. 업로드한 테스트 이미지 전체 예측


In [ ]:
results = []

for image_file in test_image_files:
    x = prepare_image(image_file)
    pred = model.predict(x, verbose=0)[0]
    best_idx = int(np.argmax(pred))

    row = {
        "file": image_file,
        "prediction": class_names[best_idx],
        "confidence": float(pred[best_idx])
    }
    results.append(row)
    print(row)


## 실습 7. 결과를 표로 확인


In [ ]:
import pandas as pd

result_df = pd.DataFrame(results)
result_df


## 실습 8. 학습 결과 기록

아래 항목을 Notion 워크북 표에도 옮겨 적으세요.

- 실제 클래스
- 예측 클래스
- confidence
- 성공/실패
- 오분류 원인 가설

### 생각해보기
1. confidence가 높으면 항상 정답인가?
2. 배경, 조명, 거리 변화가 결과에 어떤 영향을 주는가?
3. 내가 만든 Teachable Machine 모델을 다른 Python 환경에서 재사용할 수 있다는 것은 무엇을 의미하는가?
